# AI-103 Module 2: one deployment, four setup decisions

**Lesson 2: Set up AI solutions in Foundry.**

Module 2 is four **setup** decisions: how to **design the infrastructure**, which **deployment option** to pick, how to **configure** the deployment, and how to fold it into **CI/CD**. Rather than four disconnected snippets, this notebook builds the **one** artifact those decisions produce: a real, running model deployment, then tears it back down.

Everything is **keyless** (Microsoft Entra ID, no application programming interface (API) keys). We authenticate with `AzureCliCredential`, which binds to your `az login` identity, so the demo is deterministic even when service-principal environment variables are set for other tools. In a pipeline you swap that one line for `DefaultAzureCredential` and let a federated (OIDC) identity resolve (LO4).

Each learning objective (LO) maps to one property of the deployment:

| LO | Decision | Where it shows up |
|---|---|---|
| **LO1** | design the **infrastructure** | keyless auth to a hub-less Foundry **project** + its parent **account** |
| **LO2** | choose a **deployment option** | `sku.name` (`GlobalStandard` = pay-per-token) |
| **LO3** | **configure** the deployment | `versionUpgradeOption` (pin) + `raiPolicyName` (content filter) |
| **LO4** | integrate with **CI/CD** | `begin_create_or_update` is an idempotent, secretless upsert |

**Two Azure planes, and telling them apart is an exam skill:**
- **Control plane** (`azure-mgmt-cognitiveservices`) **creates/configures** the deployment. Needs **Cognitive Services Contributor** on the account.
- **Data plane** (`azure-ai-projects`) **reads and calls** it. Needs **Foundry User** on the project. Plain Owner/Contributor grant no data actions, a classic AI-103 trap.

**Before you run:** `uv sync`, then `az login`, then copy `.env.example` to `.env` and fill in your Foundry values. Run the cells **top to bottom**.

## Setup: imports and the two-plane clients

We import both software development kits (SDKs): the **control-plane** management client that creates and configures the deployment, and the **data-plane** projects client that reads and calls it. One credential (`AzureCliCredential`) drives both. We load `.env` for convenience; real environment variables or a pipeline's secretless identity work identically.

In [1]:
import os

from azure.identity import AzureCliCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient  # CONTROL plane
from azure.ai.projects import AIProjectClient                              # DATA plane

try:
    from dotenv import load_dotenv
    load_dotenv()  # reads .env next to this notebook; copy from .env.example first
except ImportError:
    pass  # .env is a convenience; real env vars work too.


def required_env(name: str, hint: str) -> str:
    """Read an environment variable or raise with a copy-pasteable hint.

    A missing endpoint, subscription, or account name is the number-one reason a first
    run fails, so we fail fast with a clear message instead of a deep stack trace.
    """
    value = os.environ.get(name)
    if not value:
        raise SystemExit(f"ERROR: {name} is not set.\n  -> {hint}\n  See .env.example.")
    return value


print("Imports OK. Control-plane + data-plane SDKs ready.")

Imports OK. Control-plane + data-plane SDKs ready.


## LO1: design the infrastructure

The infrastructure decision is where and how the deployment lives, and who may touch it. We connect **keyless** to a **hub-less Foundry project** (the modern default, no AI Hub required) and its parent **Foundry account**. One `AzureCliCredential` builds both clients: the management client on the account (control plane) and the projects client on the project endpoint (data plane). No API keys, exactly the AI-103 security posture. (Docs: [Foundry RBAC](https://learn.microsoft.com/azure/foundry/concepts/rbac-foundry).)

In [2]:
# LO1 -- the INFRASTRUCTURE: one credential, two planes.
project_endpoint = required_env(
    "FOUNDRY_PROJECT_ENDPOINT",
    "https://<resource>.services.ai.azure.com/api/projects/<project>",
)
subscription_id = required_env("AZURE_SUBSCRIPTION_ID", "your Azure subscription GUID")
resource_group = required_env("AZURE_RESOURCE_GROUP", "the resource group holding the Foundry account")
account_name = required_env("FOUNDRY_ACCOUNT_NAME", "the Foundry (Cognitive Services) account name")

# AzureCliCredential pins to your `az login` identity. DefaultAzureCredential would check
# EnvironmentCredential first, so a stray AZURE_CLIENT_ID/SECRET could silently sign you in
# as a service principal instead. In CI/CD you swap this one line for DefaultAzureCredential.
credential = AzureCliCredential()
arm = CognitiveServicesManagementClient(credential, subscription_id)   # CONTROL plane (create/config)
project = AIProjectClient(endpoint=project_endpoint, credential=credential)  # DATA plane (read/call)

print(f"LO1  account (control plane) : {account_name}  (rg: {resource_group})")
print(f"LO1  project (data plane)    : {project_endpoint}")
print("LO1  auth                    : AzureCliCredential (az login) -- no API keys")
print("     EXAM TELL: control-plane create needs Cognitive Services Contributor;")
print("                data-plane read/call needs Foundry User. Owner/Contributor grant no data actions.")

LO1  account (control plane) : aif-ai103-core-quc6kxy2j2v2q  (rg: rg-ai103-core)
LO1  project (data plane)    : https://aif-ai103-core-quc6kxy2j2v2q.services.ai.azure.com/api/projects/proj-lesson-02
LO1  auth                    : AzureCliCredential (az login) -- no API keys
     EXAM TELL: control-plane create needs Cognitive Services Contributor;
                data-plane read/call needs Foundry User. Owner/Contributor grant no data actions.


## LO2 + LO3: choose a deployment option and configure it

Now the two decisions that ride on the deployment **spec**. **LO2** is `sku.name`: the whole elastic-vs-reserved-vs-batch choice compressed into one string. `GlobalStandard` is pay-per-token shared capacity, the recommended starting point; swap to `ProvisionedManaged` for reserved PTU throughput, or `GlobalBatch` for async 50%-off jobs.

**LO3** is two knobs the exam tests by name: `versionUpgradeOption = NoAutoUpgrade` **pins** the model version so a vendor update can't silently change behavior, and `raiPolicyName` attaches the **content-filter** (responsible-AI) policy at the deployment. We build a plain dict, not typed classes, because it mirrors the Bicep in `../../infra/ai103-core.bicep` one-for-one, which is the LO4 bridge to infrastructure as code. (Docs: [deployment types](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/deployment-types), [deployment template fields](https://learn.microsoft.com/azure/templates/microsoft.cognitiveservices/accounts/deployments).)

In [3]:
# LO2 (sku) + LO3 (version pin + content filter): assemble the deployment spec.
DEPLOYMENT_NAME = os.environ.get("DEPLOY_NAME", "ai103-l2-demo")
SKU_NAME = os.environ.get("DEPLOY_SKU", "GlobalStandard")            # LO2: the deployment "shape"
SKU_CAPACITY = int(os.environ.get("DEPLOY_CAPACITY", "1"))          # tiny on purpose
VERSION_UPGRADE_OPTION = os.environ.get("DEPLOY_UPGRADE_OPTION", "NoAutoUpgrade")   # LO3: pin
CONTENT_FILTER_POLICY = os.environ.get("DEPLOY_RAI_POLICY", "Microsoft.DefaultV2")  # LO3: filter

model_block = {
    "format": os.environ.get("DEPLOY_MODEL_FORMAT", "OpenAI"),
    "name": required_env("DEPLOY_MODEL_NAME", "the base model to deploy, e.g. gpt-5-nano"),
}
model_version = os.environ.get("DEPLOY_MODEL_VERSION", "")
if model_version:  # some model families (e.g. gpt-5) REQUIRE an explicit version -- itself the LO3 lesson
    model_block["version"] = model_version

spec = {
    "sku": {"name": SKU_NAME, "capacity": SKU_CAPACITY},                 # LO2
    "properties": {
        "model": model_block,
        "versionUpgradeOption": VERSION_UPGRADE_OPTION,                  # LO3 -- pin
        "raiPolicyName": CONTENT_FILTER_POLICY,                          # LO3 -- content filter
    },
}

print(f"LO2  sku.name             : {spec['sku']['name']}  (GlobalStandard = pay-per-token shared)")
print(f"     sku.capacity         : {spec['sku']['capacity']}")
print(f"LO3  versionUpgradeOption : {spec['properties']['versionUpgradeOption']}  (NoAutoUpgrade = pinned)")
print(f"     raiPolicyName        : {spec['properties']['raiPolicyName']}  (content filter policy)")
print(f"     model                : {spec['properties']['model']}")

LO2  sku.name             : GlobalStandard  (GlobalStandard = pay-per-token shared)
     sku.capacity         : 1
LO3  versionUpgradeOption : NoAutoUpgrade  (NoAutoUpgrade = pinned)
     raiPolicyName        : Microsoft.DefaultV2  (content filter policy)
     model                : {'format': 'OpenAI', 'name': 'gpt-5-nano', 'version': '2025-08-07'}


### Deploy it (the idempotent upsert that is also the LO4 seed)

`begin_create_or_update` is an **idempotent upsert**: run it once and Azure builds the deployment; run it again and Azure converges to the same desired state, no harm done. That property is exactly what makes it safe to drop into a pipeline. `.result()` blocks until the deployment reaches a terminal state.

In [4]:
# LO2/LO3 executed: create-or-update. Idempotent -- safe to re-run this cell.
print(f"Deploying '{DEPLOYMENT_NAME}' ... (create-or-update; safe to re-run)")
result = arm.deployments.begin_create_or_update(
    resource_group_name=resource_group,
    account_name=account_name,
    deployment_name=DEPLOYMENT_NAME,
    deployment=spec,
).result()  # blocks until the deployment reaches a terminal state
print("provisioningState:", result.properties.provisioning_state)

Deploying 'ai103-l2-demo' ... (create-or-update; safe to re-run)


provisioningState: Succeeded


## LO3: verify the configuration stuck, then call it

Now the **data plane proves the control plane**. We read the deployment's settings straight back from the management client, so the pinned version and content-filter policy are sitting exactly where we left them. Then we confirm the same deployment is visible through the projects client and send it one tiny prompt. Portal and code agree, because they were always the same room.

In [5]:
# LO3 proof #1: read the config BACK from the control plane.
live = arm.deployments.get(resource_group, account_name, DEPLOYMENT_NAME)
print("sku.name (read back)     :", live.sku.name)
print("model.version (read back):", getattr(live.properties.model, "version", "(service default)"))
print("versionUpgradeOption     :", live.properties.version_upgrade_option, " (NoAutoUpgrade = pinned)")
print("raiPolicyName            :", live.properties.rai_policy_name)

# LO3 proof #2: the DATA plane sees the same deployment (never fatal if listing is unavailable).
try:
    seen = next((d for d in project.deployments.list() if d.name == DEPLOYMENT_NAME), None)
    if seen is not None:
        print(f"data-plane sees it       : {seen.name}  model={seen.model_name}  "
              f"v={seen.model_version}  type={seen.type}")
except Exception as exc:
    print(f"(data-plane list skipped : {type(exc).__name__})")

# LO3 proof #3: actually CALL the deployment.
try:
    with project.get_openai_client() as openai_client:
        ping = openai_client.responses.create(
            model=DEPLOYMENT_NAME,
            input="Reply with the single word: ready.",
        )
        print("live inference           :", f"'{ping.output_text.strip()}'  (deployment is callable)")
except Exception as exc:
    # Non-OpenAI models may not speak the Responses API; provisioningState already proved it.
    print(f"live inference           : skipped ({type(exc).__name__}); provisioningState confirms it")

sku.name (read back)     : GlobalStandard
model.version (read back): 2025-08-07
versionUpgradeOption     : NoAutoUpgrade  (NoAutoUpgrade = pinned)
raiPolicyName            : Microsoft.DefaultV2


data-plane sees it       : ai103-l2-demo  model=gpt-5-nano  v=2025-08-07  type=ModelDeployment


live inference           : 'ready'  (deployment is callable)


## LO4: integrate with CI/CD

Everything above is repeatable and secretless, which is the whole CI/CD story. The `spec` dict maps 1:1 to the Bicep resource in `../../infra/ai103-core.bicep` (`sku.name`, `model`, `versionUpgradeOption`, `raiPolicyName`), so "treat AI infrastructure as code" is literal here. Re-running the create cell converges to the same state, and in GitHub Actions you swap `AzureCliCredential` for `DefaultAzureCredential` so `azure/login` with `id-token: write` supplies a federated (OIDC) identity, no stored secret to leak.

In [6]:
# LO4 -- the CI/CD contract, stated as data so it is easy to recite on camera.
cicd_facts = {
    "idempotent": "begin_create_or_update converges to the same state on every run",
    "infra-as-code": "spec keys map 1:1 to infra/ai103-core.bicep",
    "secretless": "swap AzureCliCredential -> DefaultAzureCredential; OIDC federated identity in Actions",
}
for k, v in cicd_facts.items():
    print(f"LO4  {k:14}: {v}")

LO4  idempotent    : begin_create_or_update converges to the same state on every run
LO4  infra-as-code : spec keys map 1:1 to infra/ai103-core.bicep
LO4  secretless    : swap AzureCliCredential -> DefaultAzureCredential; OIDC federated identity in Actions


## Recap and cleanup

One deployment, four setup decisions: the **infrastructure** and keyless auth (LO1), the **deployment option** in `sku.name` (LO2), the **configuration** in `versionUpgradeOption` + `raiPolicyName` (LO3), and the **idempotent, secretless upsert** that fits a pipeline (LO4). The last cell deletes the deployment so the demo leaves nothing billing. In real work you would keep it standing; here we tidy up. The cell is safe to re-run even when nothing is left.

In [7]:
# Tidy up so the meter stops. Idempotent: deleting an absent deployment is not an error here.
try:
    arm.deployments.begin_delete(resource_group, account_name, DEPLOYMENT_NAME).result()
    print(f"deleted: {DEPLOYMENT_NAME} -- nothing left running.")
except Exception as exc:
    print(f"nothing to delete ({type(exc).__name__}); '{DEPLOYMENT_NAME}' was already gone.")

deleted: ai103-l2-demo -- nothing left running.
